In [7]:
import pandas as pd
import numpy as np
import mysql.connector
from mysql.connector import Error

# Connect to MySQL database
def get_db_connection():
    mydb = mysql.connector.connect(
        host="alvcantu.mysql.pythonanywhere-services.com",
        user="alvcantu",
        password="h63Efp09-d",
        database="alvcantu$default"
    )
    cursor = mydb.cursor()
    return mydb, cursor


sql_query_extract = '''
SELECT * FROM BM_FactCustomers;
'''

# Fetch data
mydb, cursor = get_db_connection()
cursor.execute(sql_query_extract)
data = cursor.fetchall()
columns = [i[0] for i in cursor.description]  # Get column names

# Convert to DataFrame
df = pd.DataFrame(data, columns=columns)

# Convert ENUM and other categorical data to category type for better memory usage and performance
for col in df.columns:
    if df[col].dtype == 'object':  # This will catch both ENUM and VARCHAR
        df[col] = df[col].astype('category')

In [8]:
# Assuming 'duration' should not be used in the model as per following documentation:
# Duration: last contact duration, in seconds (numeric). Important
# note: this attribute highly affects the output target (e.g., if
# duration=0 then y='no'). Yet, the duration is not known before a call
# is performed. Also, after the end of the call y is obviously known.
# Thus, this input should only be included for benchmark purposes and
# should be discarded if the intention is to have a realistic
# predictive model.
features = df.drop(columns=['subscribed_y', 'duration','customer_id'])
target = df['subscribed_y']

In [9]:
print(features.dtypes)

age                  int64
job               category
marital           category
education         category
default_credit    category
balance            float64
housing           category
loan              category
contact           category
month             category
campaign             int64
pdays              float64
previous             int64
poutcome          category
dtype: object


In [5]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split


def encode_categorical_columns(df, encoding_type='label'):
    """
    Encodes categorical columns in the dataframe either using LabelEncoder or One-Hot Encoding.

    Parameters:
    df (pd.DataFrame): The dataframe containing the features.
    encoding_type (str): The type of encoding to apply ('label' for LabelEncoder, 'onehot' for One-Hot Encoding).

    Returns:
    pd.DataFrame: The dataframe with encoded categorical columns.
    """
    # Make a copy of the dataframe to avoid modifying the original
    df_encoded = df.copy()
    
    # Get the list of categorical columns
    categorical_columns = df_encoded.select_dtypes(include=['category', 'object']).columns
    
    # Loop through each categorical column and encode it
    if encoding_type == 'label':
        # Label Encoding (converts categories to integers)
        le = LabelEncoder()
        for column in categorical_columns:
            df_encoded[column] = le.fit_transform(df_encoded[column])
    
    elif encoding_type == 'onehot':
        # One-Hot Encoding (converts categories to binary columns)
        df_encoded = pd.get_dummies(df_encoded, columns=categorical_columns)
    
    else:
        raise ValueError("Invalid encoding_type. Choose 'label' or 'onehot'.")
    
    return df_encoded

# Encode categorical variables
le = LabelEncoder()
for column in features.select_dtypes(include=['category', 'object']):
    features[column] = le.fit_transform(features[column])

# Set train and test data
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.2, random_state=42)

In [6]:
# Temporary compatibility function
if not hasattr(np, 'round_'):
    np.round_ = np.round

if not hasattr(np, 'unicode_'):
    np.unicode_ = np.str_

from lazypredict.Supervised import LazyClassifier
# Determine what model to use with laxy classifier
# Initialize LazyClassifier
clf = LazyClassifier(verbose=0, ignore_warnings=True, custom_metric=None)
# Fit on the train set
models, predictions = clf.fit(X_train, X_test, y_train, y_test)
# This will take some time to run as it tests many different models
print(models)

AttributeError: module 'pandas.core.strings' has no attribute 'StringMethods'